Load Dataset (Chunk-wise loading for large CSVs)

The CICDDoS2019 dataset is HUGE.
So we load it in chunks, process, then concatenate.

In [ ]:
import pandas as pd

def load_dataset(filepath, chunksize=50000):
    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=chunksize):
        chunks.append(chunk)
    df = pd.concat(chunks, axis=0)
    return df

df = load_dataset("CICDDoS2019.csv")
print(df.shape)
print(df.head())

Basic Cleaning

Common issues in CICDDoS2019:

    Infinite values

    Missing values

    Non-numeric features

In [ ]:
import numpy as np

# Replace inf values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop columns with too many missing values
df.dropna(axis=1, thresh=len(df)*0.6, inplace=True)

# Fill remaining missing values
df.fillna(0, inplace=True)

# Convert label column to numeric
df['Label'] = df['Label'].astype('category').cat.codes

df.info()


Feature Selection / Removing Useless Columns

    Some columns add no value (Flow ID, Timestamp, etc.)

In [ ]:
cols_to_drop = ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)


Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('Label', axis=1)
y = df['Label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
